 如果想要封装历史记录，除了自行维护历史消息外，也可以借助LangChain内置的历史记录附加功能。
LangChain提供History功能，帮助模型在有历史记忆的情况下回答。
· 基于RunnableWithMessageHistory在原有链的基础上创建带有历史记录功能的新链(新Runnable实例)
· 基于InMemoryChatMessageHistory为历史记录提供内存存储(临时用)

In [4]:
from jupyter_server.pytest_plugin import some_resource
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_models.tongyi import ChatTongyi
from langsmith.evaluation import _name_generation

# 通过RunnableWithMessageHistory获取一个新的带有历史记录功能的chain
conversation_chains=RunnableWithMessageHistory(
    some_resource,        #被附加历史消息的Runnable，通常是chain
    None,            #获取指定会话ID的历史会话的函数
    input_messages_key="input",        #声明用户输入消息在模板中的占位符
    history_messages_key="chat_history"        #声明历史消息在模板中的占位符
)

# 获取指定会话ID的历史会话记录函数
chat_history_store={}    # 存放多个会话ID所对应的历史会话记录
# 函数传入为会话ID （字符串类型）
# 函数要求返回BaseChatMessageHistory的子类
# BaseChatMessageHistory类专用于存放某个会话的历史记录
# InMemoryChatMessageHistory是官方自带的基于内存存放历史记录的类
def get_history(session_id):
    if session_id not in chat_history_store:
        # 返回一个新的实例
        chat_history_store[session_id]=InMemoryChatMessageHistory()
        return chat_history_store[session_id]


model=ChatTongyi(model="qwen3-max")
prompt=PromptTemplate.from_template(
    "根据会话历史，回应用户问题。对话历史：{chat_history}，用户提问：{input}，请回答"
)
str_parser=StrOutputParser
base_chain=prompt|model|str_parser

#创建一个新的链，对原有链增强功能：自动附加历史消息
conversation_chain=RunnableWithMessageHistory(
    base_chain,     #被增强的原有chain
    get_history,    #通过会话id获取InMemoryChatMessageHistory类对象
    input_messages_key="input",     #表示用户输入再模板中的占位符
    history_messages_key="chat_history"      #表示历史消息在模板中的占位符
)

if _name_generation == '_main_':
    # 固定格式，添加到LangChain的配置，为当前程序配置所属的session_id
    session_config={
        "configuable":{
            "session_id":"user_001",
        }
    }
    conversation_chain.invoke({"input":"小明有两只猫"},session_config)

ModuleNotFoundError: No module named 'pytest'